<a href="https://colab.research.google.com/github/Drewbits/petrophysical-data-quality-workflow/blob/main/notebooks/01_Inventory_and_Explore_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Petrophysical Data Quality Workflow

## Notebook 1 – Load and Inspect LAS Files

### Objective

#This notebook demonstrates the first stage of a petrophysical data quality workflow.
#The goal is to load LAS files, inspect their metadata and curve information, and
#prepare the data for subsequent quality control, standardization, and normalization.

### Learning Objectives

#- Understand the structure of LAS files
#- Read well headers and metadata
#- Inspect curve mnemonics and units
#- Load log data into a pandas DataFrame
#- Verify data integrity before processing

In [1]:
# Install required packages
!pip install lasio pandas matplotlib numpy

# New Section

In [2]:
from pathlib import Path
import lasio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display DataFrames more cleanly
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from pathlib import Path

DATASET_ROOT = Path(
    "/content/drive/MyDrive/Datasets/15_9-19 A/04.COMPOSITE"
)

print("Dataset exists:", DATASET_ROOT.exists())
print("Dataset root:", DATASET_ROOT)

Dataset exists: True
Dataset root: /content/drive/MyDrive/Datasets/15_9-19 A/04.COMPOSITE


In [5]:
all_files = [
    path
    for path in DATASET_ROOT.rglob("*")
    if path.is_file()
]

print(f"Files found: {len(all_files):,}")

Files found: 104


In [6]:
inventory_df = pd.DataFrame(
    {
        "file_name": [path.name for path in all_files],
        "extension": [
            path.suffix.upper() if path.suffix else "[NO EXTENSION]"
            for path in all_files
        ],
        "relative_path": [
            str(path.relative_to(DATASET_ROOT))
            for path in all_files
        ],
        "parent_folder": [
            path.parent.name
            for path in all_files
        ],
        "size_mb": [
            path.stat().st_size / 1_000_000
            for path in all_files
        ],
    }
)

inventory_df.head()

,file_name,extension,relative_path,parent_folder,size_mb
0,12.tif,.TIF,15_9-19 A/12.tif,15_9-19 A,3.441363
1,L898AUD.ASC,.ASC,15_9-19 A/L898AUD.ASC,15_9-19 A,0.059812
2,L898CMP.LTI,.LTI,15_9-19 A/L898CMP.LTI,15_9-19 A,6.488411
3,read me.txt,.TXT,15_9-19 A/read me.txt,15_9-19 A,0.000085
4,Thumbs.db,.DB,15_9-19 A/Thumbs.db,15_9-19 A,0.005120


In [7]:
print("Files found:", len(all_files))

Files found: 104


In [8]:
file_type_summary = (
    inventory_df.groupby("extension")
    .agg(
        file_count=("file_name", "count"),
        total_size_mb=("size_mb", "sum"),
    )
    .sort_values("file_count", ascending=False)
    .reset_index()
)

file_type_summary["total_size_mb"] = file_type_summary["total_size_mb"].round(2)

file_type_summary

,extension,file_count,total_size_mb
0,.PDF,42,5.88
1,.DLIS,34,106.54
2,.ASC,9,0.16
3,.LIS,8,16.83
4,.TXT,3,0.00
5,.LTI,3,16.29
6,.LAS,2,4.20
7,.DB,1,0.01
8,.TIF,1,3.44
9,[NO EXTENSION],1,0.00


In [9]:
log_extensions = [".LAS", ".DLIS", ".LIS", ".ASC"]

log_files_df = inventory_df[
    inventory_df["extension"].isin(log_extensions)
].copy()

print(f"Potential digital log files: {len(log_files_df)}")

log_files_df.head(20)

Potential digital log files: 53


,file_name,extension,relative_path,parent_folder,size_mb
1,L898AUD.ASC,.ASC,15_9-19 A/L898AUD.ASC,15_9-19 A,0.059812
5,L916AUD.ASC,.ASC,15_9-19 BT2/L916AUD.ASC,15_9-19 BT2,0.020495
7,L916CON.ASC,.ASC,15_9-19 BT2/L916CON.ASC,15_9-19 BT2,0.013512
8,L916HEAD.ASC,.ASC,15_9-19 BT2/L916HEAD.ASC,15_9-19 BT2,0.014265
9,L916INFO.ASC,.ASC,15_9-19 BT2/L916INFO.ASC,15_9-19 BT2,0.000814
11,L749AUD.ASC,.ASC,15_9-19 SR/L749AUD.ASC,15_9-19 SR,0.022530
13,L749CON.ASC,.ASC,15_9-19 SR/L749CON.ASC,15_9-19 SR,0.013371
14,L749HEAD.ASC,.ASC,15_9-19 SR/L749HEAD.ASC,15_9-19 SR,0.012789
15,L749INFO.ASC,.ASC,15_9-19 SR/L749INFO.ASC,15_9-19 SR,0.000832
17,STAT1990__30-1__15-9-19_SR__COMPOSITE__1.LAS,.LAS,15_9-19 SR/STAT1990__30-1__15-9-19_SR__COMPOSI...,15_9-19 SR,2.621599


I began by recursively inventorying the archive rather than assuming its contents. I captured file paths, formats, and sizes in a structured DataFrame, then summarized the archive by extension to determine which ingestion methods would be required.

In [10]:
output_folder = Path("/content/drive/MyDrive/Datasets/volve_dataset_outputs")
output_folder.mkdir(exist_ok=True)

inventory_path = output_folder / "volve_file_inventory.csv"
summary_path = output_folder / "volve_file_type_summary.csv"

inventory_df.to_csv(inventory_path, index=False)
file_type_summary.to_csv(summary_path, index=False)

print(f"Saved: {inventory_path}")
print(f"Saved: {summary_path}")

Saved: /content/drive/MyDrive/Datasets/volve_dataset_outputs/volve_file_inventory.csv
Saved: /content/drive/MyDrive/Datasets/volve_dataset_outputs/volve_file_type_summary.csv
